In [16]:
import threading
import json
import smtplib
from jinja2 import Template
from google.cloud import bigquery
from datetime import datetime
import time
from google.api_core.exceptions import BadRequest
def run_query(template_file, params):
    try:
        with open(template_file, 'r') as file:
            template_content = file.read()
        template = Template(template_content)
        rendered_sql = template.render(**params)
        print(f"Running query from {template_file}...")
        start_time = time.time()
        query_job = client.query(rendered_sql)
        query_job.result()
        end_time = time.time()
        query_duration=end_time-start_time
        print(f"Query from {template_file} completed successfully - duration: {query_duration:.2f} seconds")
    except BadRequest as e:
        print(f"Query from {template_file} failed with error: {e}")
        return False
        
    except Exception as e:
        print(f"Query from {template_file} failed with error: {e}")
        return False
    return True

def run_parallel_query(template_files, params):
    threads = []
    results = []
    def thread_func(template):
        result = run_query(template,params)
        results.append(result)

    for template_file in template_files:
        thread = threading.Thread(target=thread_func, args=(template_file,))
        threads.append(thread)
        thread.start()
    for thread in threads:
        thread.join()
    
    return all(results)


In [21]:
client = bigquery.Client(project='anbc-hcb-dev')
# read params from json
with open('config.json', 'r') as file:
    config = json.load(file)
USER = "zhongc"
PROJECT_ID = 'anbc-hcb-dev'
ENV = "dev"

COSTCENTER = config['config']['COSTCENTER']
DATASET = config['config']['DATASET'].format(ENV=ENV)
DEC_DATASET = config['config']['DEC_DATASET'].format(ENV=ENV)
owner_name = f"{USER}_aetna_com"
databases = config['config']['bqdataset']
databases = {x:databases[x].format(PROJECT_ID=PROJECT_ID,DATASET=DATASET,DEC_DATASET=DEC_DATASET) for x in databases}
databases["DATASET"] = DATASET
databases["DEC_DATASET"] = DEC_DATASET
params = databases
params['COSTCENTER'] = COSTCENTER
params['OWNER'] = owner_name
params['project_name'] = PROJECT_ID
params["PREFIX"]="me_rap_daily_clm_transformer"
# current_dt = datetime.now()
# current_dt = current_dt.strftime('%Y-%m-%d')
# current_dt='2023-05-16'
# params['index_dt']=current_dt

In [22]:
params

{'TARGET_DB': 'anbc-hcb-dev.clin_analytics_hcb_dev',
 'DEC_TARGET_DB': 'anbc-hcb-dev.clin_analytics_hcb_dev',
 'DATASET': 'clin_analytics_hcb_dev',
 'DEC_DATASET': 'clin_analytics_hcb_dev',
 'COSTCENTER': '13070',
 'OWNER': 'zhongc_aetna_com',
 'project_name': 'anbc-hcb-dev',
 'PREFIX': 'me_rap_daily_clm_transformer'}

In [31]:
def main_func():
    # if not run_query('01_member.sql', params): return
    
    # jobs=['02_a_edw_claims.sql','02_b_insights_clms.sql','02_c_rx_cur_archive_dec.sql','02_d_edw_archive_clms_dec.sql']
    # if not run_parallel_query(jobs, params): return
    
    # if not run_query('02_a_edw_claims.sql', params): return

    if not run_query('03_combine_edw_insights_dec.sql', params): return
    if not run_query('04_combine_archive_dec.sql', params): return 
    
    jobs=['05_a_add_icd_dec.sql','05_b_add_icd_archive_dec.sql']
    if not run_parallel_query(jobs, params): return

    if not run_query('06_icd_archive_combine_dec.sql', params): return
    if not run_query('07_prep_transformer_a_dec.sql', params): return


In [32]:
main_func()

Running query from 03_combine_edw_insights_dec.sql...
Query from 03_combine_edw_insights_dec.sql completed successfully - duration: 55.91 seconds
Running query from 04_combine_archive_dec.sql...
Query from 04_combine_archive_dec.sql completed successfully - duration: 43.38 seconds
Running query from 05_a_add_icd_dec.sql...
Running query from 05_b_add_icd_archive_dec.sql...
Query from 05_b_add_icd_archive_dec.sql completed successfully - duration: 26.38 seconds
Query from 05_a_add_icd_dec.sql completed successfully - duration: 154.65 seconds
Running query from 06_icd_archive_combine_dec.sql...
Query from 06_icd_archive_combine_dec.sql completed successfully - duration: 40.93 seconds
Running query from 07_prep_transformer_a_dec.sql...
Query from 07_prep_transformer_a_dec.sql completed successfully - duration: 168.03 seconds
